In [ ]:
%matplotlib inline
import nest_asyncio
nest_asyncio.apply()

import matplotlib
matplotlib.rcParams['font.family'] = 'monospace'
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False
matplotlib.rcParams['figure.facecolor'] = '#fafafa'
matplotlib.rcParams['axes.facecolor'] = '#fafafa'

# Anomaly Detection — Precios de Gasolina España

Detección de anomalías en la serie temporal de precios semanales nacionales usando **IQR** e **Isolation Forest**.

In [ ]:
from sklearn.ensemble import IsolationForest

# Isolation Forest sobre la serie de medias nacionales
X = media_nacional.values.reshape(-1, 1)

iso = IsolationForest(contamination=0.05, random_state=42)
etiquetas = iso.fit_predict(X)  # -1 = anomalia, 1 = normal

anomalias_if = media_nacional[etiquetas == -1]

print('Isolation Forest (contamination=0.05)')
print('--------------------------------------')
print('Semanas anomalas ({} detectadas):'.format(len(anomalias_if)))
for semana, precio in anomalias_if.sort_index().items():
    print('  {} -> media nacional = {:.4f} euro/L'.format(semana, precio))

print()
print('Comparacion IQR vs Isolation Forest')
print('------------------------------------')
set_iqr = set(anomalias_iqr.index)
set_if = set(anomalias_if.index)
acuerdo = set_iqr & set_if
solo_iqr = set_iqr - set_if
solo_if = set_if - set_iqr

print('Acuerdo (ambos):  {}'.format(sorted(acuerdo)))
print('Solo IQR:         {}'.format(sorted(solo_iqr)))
print('Solo IF:          {}'.format(sorted(solo_if)))

## 2. Isolation Forest sobre la serie temporal de medias nacionales

Usamos `IsolationForest(contamination=0.05)` de scikit-learn sobre la misma serie de 52 medias semanales.
Comparamos con los resultados del IQR.

In [ ]:
# Serie temporal de medias nacionales semanales
media_nacional = df.mean(axis=1)  # Serie de 52 valores (una media por semana)

# IQR sobre la serie temporal (no cross-province)
Q1 = media_nacional.quantile(0.25)
Q3 = media_nacional.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Semanas anomalas segun IQR
anomalias_iqr = media_nacional[(media_nacional < limite_inferior) | (media_nacional > limite_superior)]

print('IQR sobre serie temporal de medias nacionales')
print('---------------------------------------------')
print('Q1:               {:.4f}'.format(Q1))
print('Q3:               {:.4f}'.format(Q3))
print('IQR:              {:.4f}'.format(IQR))
print('Limite inferior:  {:.4f}'.format(limite_inferior))
print('Limite superior:  {:.4f}'.format(limite_superior))
print()
print('Semanas anomalas ({} detectadas):'.format(len(anomalias_iqr)))
for semana, precio in anomalias_iqr.items():
    print('  {} -> media nacional = {:.4f} euro/L'.format(semana, precio))

## 1. Detección IQR sobre la serie temporal de medias nacionales

Calculamos la media nacional por semana y aplicamos IQR sobre esa serie de 52 puntos.
Una semana es outlier si su media cae fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# 50 provincias espanolas
provincias = [
    'Alava', 'Albacete', 'Alicante', 'Almeria', 'Asturias',
    'Avila', 'Badajoz', 'Barcelona', 'Burgos', 'Caceres',
    'Cadiz', 'Cantabria', 'Castellon', 'Ciudad Real', 'Cordoba',
    'Cuenca', 'Girona', 'Granada', 'Guadalajara', 'Guipuzcoa',
    'Huelva', 'Huesca', 'Jaen', 'La Rioja', 'Las Palmas',
    'Leon', 'Lleida', 'Lugo', 'Madrid', 'Malaga',
    'Murcia', 'Navarra', 'Orense', 'Palencia', 'Pontevedra',
    'Salamanca', 'Santa Cruz de Tenerife', 'Segovia', 'Sevilla', 'Soria',
    'Tarragona', 'Teruel', 'Toledo', 'Valencia', 'Valladolid',
    'Vizcaya', 'Zamora', 'Zaragoza', 'Ceuta', 'Melilla'
]

n_provincias = len(provincias)  # 50
n_semanas = 52

# Precios base gasolina95: media ~1.55 con ligera tendencia
tendencia = np.linspace(1.52, 1.58, n_semanas)

# Ruido por provincia (variacion estructural fija)
ruido_provincia = np.random.normal(0, 0.03, n_provincias)

# Precio base por semana x provincia
precios = np.zeros((n_semanas, n_provincias))
for i in range(n_semanas):
    ruido_semana = np.random.normal(0, 0.008, n_provincias)
    precios[i, :] = tendencia[i] + ruido_provincia + ruido_semana

# Inyectar 3 picos nacionales (semanas 10, 28, 42)
for semana_pico in [10, 28, 42]:
    precios[semana_pico, :] += np.random.uniform(0.10, 0.15, n_provincias)

# Inyectar outlier cronico: Soria (indice 39) +0.12 todas las semanas
idx_soria = provincias.index('Soria')
precios[:, idx_soria] += 0.12

# Crear DataFrame
semanas = ['S{:02d}'.format(i+1) for i in range(n_semanas)]
df = pd.DataFrame(precios, index=semanas, columns=provincias)

print('Shape:', df.shape)
print('\nEstadisticas generales:')
print(df.describe().round(4))
print('\nMedia por semana (primeras 5):')
print(df.mean(axis=1).head())